# PVGIS ST-GNN + SDE-Net: training diretto multi-orizzonte t+1…t+6

Un **solo training** produce sei canali di output, uno per ogni ora futura da t+1 a t+6. Non vengono addestrati sei modelli separati e non viene usato rollout autoregressivo. La loss è la media della Gaussian NLL sui target `[B, N, 6]`.

Il modello è addestrato su **tutte** le finestre 2016–2018: nessuna label di anomalia entra nella loss, nella selezione delle finestre o nella normalizzazione. Gli score MTGFlow vengono passati al runner soltanto perché, con `anomaly_source='detector'`, li usa per etichettare `predictions.csv` a fine run.

L'analisi post-hoc (normale vs anomalo per orizzonte) è in `notebooks/mtgflow_pointwise_posthoc_sdenet.ipynb` e scrive in una cartella separata; questo notebook scrive solo gli output del training.

## 1. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.environ['PYTHONPATH'] = str(REPO_ROOT) + os.pathsep + os.environ.get('PYTHONPATH', '')
os.chdir(REPO_ROOT)

from physiq_pv.experiments import sde_pipeline as pipe
print('repo root:', REPO_ROOT)

## 2. Dati e score MTGFlow

In [ ]:
PVGIS_DIR = pipe.PVGIS_DIR
ANOMALY_SOURCE = 'detector'
DETECTOR = 'mtgflow'
DETECTOR_SEED = 15
DETECTOR_ROOT = Path('outputs/pvgis_mtgflow/downstream_dense') / f'seed_{DETECTOR_SEED}'
TEST_ANOMALY_SCORES = str(DETECTOR_ROOT / 'anomaly_scores.csv')
TRAIN_ANOMALY_SCORES = str(DETECTOR_ROOT / 'train_anomaly_scores.csv')

checks = {
    'PVGIS dir': Path(PVGIS_DIR).is_dir(),
    'MTGFlow test scores': Path(TEST_ANOMALY_SCORES).is_file(),
    'MTGFlow train scores': Path(TRAIN_ANOMALY_SCORES).is_file(),
}
for name, present in checks.items():
    print(('OK     ' if present else 'MISSING') + '  ' + name)
if not all(checks.values()):
    print('Eseguire prima notebooks/mtgflow_pvgis_workflow.ipynb per creare i CSV mancanti.')

## 3. Configurazione del modello diretto multi-output

In [ ]:
FORECAST_HORIZONS = (1, 2, 3, 4, 5, 6)
CONFIG = {
    **pipe.DEFAULT_CONFIG,
    'name': 'paper_faithful_gaussian_detector_mtgflow_ep60',
    'horizon': 1,  # fallback legacy; il runner usa forecast_horizons
    'forecast_horizons': ','.join(map(str, FORECAST_HORIZONS)),
    'epochs': 60, 'batch_size': 16, 'lr': 1e-4, 'lr_g': 1e-2,
    'dropout': 0.0,
    'n_sde_steps': 4, 'sigma_max': 0.5,
    'sde_sigma_initial': 0.01, 'sde_sigma_warmup_epochs': 30,
    'ood_noise_std': 2.0, 'mc_samples': 10, 'seed': 1,
    'ood_smoke_test': True, 'ood_smoke_max_samples': 2048,
    'anomaly_source': ANOMALY_SOURCE,
    'detector_regional_quantile': 0.975,
}
OUT_DIR = pipe.make_out_dir(CONFIG)
RUN_NAME = pipe.make_run_name(CONFIG)
print('horizons:', FORECAST_HORIZONS)
print('run:', RUN_NAME)
print('output:', OUT_DIR)

## 4. Un solo comando di training

In [ ]:
TRAIN_COMMAND = pipe.build_train_command(
    CONFIG, out_dir=OUT_DIR, run_name=RUN_NAME, pvgis_dir=PVGIS_DIR,
    test_anomaly_scores=TEST_ANOMALY_SCORES,
    train_anomaly_scores=TRAIN_ANOMALY_SCORES,
    device='cuda', use_wandb=True,
)
assert TRAIN_COMMAND.count('--forecast-horizons') == 1
assert TRAIN_COMMAND[TRAIN_COMMAND.index('--forecast-horizons') + 1] == '1,2,3,4,5,6'
print(' \\n  '.join(TRAIN_COMMAND))

In [ ]:
RUN_TRAINING = True
REUSE_COMPLETED_RUN = True
ALLOW_OVERWRITE = False

required_outputs = ('best_model.pt', 'predictions.csv', 'metrics_global.csv')
run_complete = all((Path(OUT_DIR) / name).is_file() for name in required_outputs)
if RUN_TRAINING and not (REUSE_COMPLETED_RUN and run_complete):
    pipe.ensure_output_dir_available(OUT_DIR, allow_overwrite=ALLOW_OVERWRITE)
    subprocess.run(TRAIN_COMMAND, check=True)
elif run_complete:
    print('[reuse] training multi-orizzonte già completo:', OUT_DIR)
else:
    print('RUN_TRAINING=False: comando non eseguito.')

## 5. Controllo del CSV

Ogni riga rappresenta una `(issue_timestamp, location, horizon_hours)`; `timestamp` è l'istante target. Si verifica solo che il training abbia prodotto tutti gli orizzonti senza duplicati; le label normale/anomalo vengono ricalcolate nel notebook post-hoc.

In [ ]:
PREDICTIONS = Path(OUT_DIR) / 'predictions.csv'
if not PREDICTIONS.is_file():
    raise FileNotFoundError(f'Mancano le predizioni: {PREDICTIONS}')
predictions = pd.read_csv(PREDICTIONS, parse_dates=['issue_timestamp', 'timestamp'])
required = {'issue_timestamp', 'timestamp', 'location', 'horizon_hours',
            'y_true', 'y_pred_mean', 'lower_pi', 'upper_pi'}
missing = required - set(predictions.columns)
if missing:
    raise ValueError(f'Colonne mancanti: {sorted(missing)}')
assert tuple(sorted(predictions['horizon_hours'].unique())) == FORECAST_HORIZONS
assert not predictions.duplicated(['issue_timestamp', 'location', 'horizon_hours']).any()
display(predictions.head(12))
display(predictions.groupby('horizon_hours').size().rename('rows').to_frame())
print('Prossimo passo: notebooks/mtgflow_pointwise_posthoc_sdenet.ipynb')